In [ ]:
# Install required libraries
!pip install -U langchain langchain-community langchain-groq langchain-text-splitters langchain-chroma langchain-classic flashrank

In [ ]:
#import packages
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


In [ ]:

GROQ_API_KEY = userdata.get('GROQ_API_KEY') # Make sure your secret is named 'GROQ_API_KEY'
llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant") # to frame answer based on conetxt and questions
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

doc_base = ["https://medium.com/@sharmankita0604/opensearch-snapshot-lifecycle-incremental-behavior-and-cost-optimization-5d1521b438c7","https://medium.com/@sharmankita0604/opensearch-cross-region-disaster-recovery-what-actually-happens-during-failover-38789e810fd5"]
# Load document
for doc in doc_base:
    loader = WebBaseLoader(doc)
    docs = loader.load()
    chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_documents(docs)
    vectorstore = Chroma.from_documents(chunks, embeddings,collection_name="opensearch_docs")
    print(f"Total chunks created: {len(chunks)}")
    print(f"Sample content from chunk 0:\n{chunks[0].page_content[:200]}...")
    print(f"Metadata check: {chunks[0].metadata}")

In [ ]:
#this cell has rerankig logic , while retrieving the data
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
compressor = FlashrankRerank()
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [ ]:

# Custom prompt template
SYSTEM_PROMPT = """You are a helpful cloud infrastructure assistant.
Answer questions using only the provided context.
If the answer is not in the context, say "I don't have that information."
Be concise and precise.

Context:
{context}"""

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT),("placeholder", "{chat_history}"),("human", "{question}")])

# Memory - keep last 5 exchanges for the ConversationalRetrievalChain
memory = ConversationBufferWindowMemory(memory_key="chat_history",k=5,return_messages=True,output_key="answer")

# Memory for the AgentExecutor (expects 'output' key)
agent_memory = ConversationBufferWindowMemory(memory_key="chat_history",k=5,return_messages=True,output_key="output")

# Build chain
chain = ConversationalRetrievalChain.from_llm(llm=llm,retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),memory=memory,return_source_documents=True,combine_docs_chain_kwargs={"prompt": prompt})

#below line is used when re-ranking will be used while fetching retrievals
chain = ConversationalRetrievalChain.from_llm(llm=llm,retriever=compression_retriever,memory=memory,return_source_documents=True,combine_docs_chain_kwargs={"prompt": prompt})

# Test conversation
def chat(question):
    result = chain.invoke({"question": question})
    print(f"Q: {question}")
    print(f"A: {result['answer']}")
    print(f"Sources: {len(result['source_documents'])} chunks")
    print(f"Memory has: {len(memory.chat_memory.messages)} messages")
    print("---")

# Test turn 1
chat("What is the snapshot lifecycle behavior in OpenSearch?")

# Test turn 2 - uses memory
chat("What is the current pricing for OpenSearch Serverless?")

# Test turn 3 - references previous answers
chat("How to reduce OpenSearch snapshot cost?")

# Inspect the memory manually
print("\nMemory contents:")
for msg in memory.chat_memory.messages:
    print(f"{type(msg).__name__}: {msg.content[:100]}")


In [ ]:
!pip install -U duckduckgo-search
!pip install -U ddgs


In [ ]:
#this cell works for agnet logic
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.tools import create_retriever_tool


# --- STEP 1: Define Tools ---
# OpenSearch Vectorstore Tool
retriever_tool = create_retriever_tool(
    vectorstore.as_retriever(),
    "search_docs",
    "Use this for ANY technical questions about Amazon OpenSearch architecture, scaling, or storage."
)

# Web Search Tool (The Fallback)
web_search = DuckDuckGoSearchRun()
web_search.description = "Use this ONLY if search_docs fails or if the question is about general news/pricing not in docs."

tools = [retriever_tool, web_search]

# --- STEP 2: The Agentic Prompt ---
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Cloud Architect. Use 'search_docs' first. If info is missing, use 'duckduckgo_search'."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"), ])

# --- STEP 3: Create the Executor ---
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=agent_memory,     verbose=True,          handle_parsing_errors=True
)


In [ ]:
#sample agnet query
result = agent_executor.invoke({"input": "What is the max storage for Aurora?"})
print(f"Agent Output: {result['output']}")
